# Week 3 — Model Implementation and Code Documentation
## Employee Attrition Prediction Using Machine Learning

This notebook implements three baseline classification models:
- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier

Detailed evaluation and cross-validation are intentionally reserved for Week 4. Hyperparameter tuning is reserved for Week 5.


In [ ]:
# 1. Import libraries
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

RAW_PATH = "https://raw.githubusercontent.com/rachitgupt04/employee-attrition-prediction/main/WA_Fn-UseC_-HR-Employee-Attrition.csv"


## 2. Load and verify the dataset

In [ ]:
try:
    df = pd.read_csv(RAW_PATH)
    print("Dataset loaded successfully.")
    print("Shape:", df.shape)
    display(df.head())
except Exception as e:
    raise RuntimeError(
        "Dataset could not be loaded. Check internet access and the GitHub Raw URL."
    ) from e

if "Attrition" not in df.columns:
    raise ValueError("Required target column 'Attrition' is missing.")


## 3. Prepare features and target

In [ ]:
data = df.copy()

# Remove identifier/constant columns used in the standard IBM HR dataset.
drop_candidates = ["EmployeeNumber", "EmployeeCount", "Over18", "StandardHours"]
drop_cols = [c for c in drop_candidates if c in data.columns]
data = data.drop(columns=drop_cols)

# Convert target to binary values.
target_map = {"No": 0, "Yes": 1}
data["Attrition"] = data["Attrition"].map(target_map)

if data["Attrition"].isna().any():
    raise ValueError("Unexpected values found in the Attrition target.")

X = data.drop(columns="Attrition")
y = data["Attrition"].astype(int)

print("Dropped columns:", drop_cols)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Target distribution:")
display(y.value_counts().rename(index={0:"No", 1:"Yes"}))


## 4. Stratified train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


## 5. Build the Week 2 preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


## 6. Logistic Regression
Logistic Regression is used as an interpretable baseline for binary classification.


In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

try:
    logistic_model.fit(X_train, y_train)
    logistic_pred = logistic_model.predict(X_test)
    print("Logistic Regression trained successfully.")
    print("Prediction shape:", logistic_pred.shape)
    print("Basic accuracy:", round(accuracy_score(y_test, logistic_pred), 4))
except Exception as e:
    raise RuntimeError("Logistic Regression training failed.") from e


## 7. Decision Tree

In [ ]:
decision_tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))
])

try:
    decision_tree_model.fit(X_train, y_train)
    tree_pred = decision_tree_model.predict(X_test)
    print("Decision Tree trained successfully.")
    print("Prediction shape:", tree_pred.shape)
    print("Basic accuracy:", round(accuracy_score(y_test, tree_pred), 4))
except Exception as e:
    raise RuntimeError("Decision Tree training failed.") from e


## 8. Random Forest

In [ ]:
random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

try:
    random_forest_model.fit(X_train, y_train)
    forest_pred = random_forest_model.predict(X_test)
    print("Random Forest trained successfully.")
    print("Prediction shape:", forest_pred.shape)
    print("Basic accuracy:", round(accuracy_score(y_test, forest_pred), 4))
except Exception as e:
    raise RuntimeError("Random Forest training failed.") from e


## 9. Small prediction sample

In [ ]:
prediction_sample = pd.DataFrame({
    "Actual": y_test.iloc[:15].map({0:"No", 1:"Yes"}).values,
    "Logistic Regression": pd.Series(logistic_pred[:15]).map({0:"No", 1:"Yes"}),
    "Decision Tree": pd.Series(tree_pred[:15]).map({0:"No", 1:"Yes"}),
    "Random Forest": pd.Series(forest_pred[:15]).map({0:"No", 1:"Yes"})
})

display(prediction_sample)


## 10. Basic implementation verification

In [ ]:
basic_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest"],
    "Basic Accuracy": [
        accuracy_score(y_test, logistic_pred),
        accuracy_score(y_test, tree_pred),
        accuracy_score(y_test, forest_pred)
    ]
}).sort_values("Basic Accuracy", ascending=False)

display(basic_results)

print(
    "Note: Accuracy here is only a basic implementation check. "
    "Precision, recall, F1-score, confusion matrix, ROC-AUC, "
    "cross-validation and detailed error analysis belong to Week 4."
)


## 11. Week 3 conclusion
The three baseline models are now implemented through a common preprocessing pipeline. The notebook verifies that training and prediction work correctly. Detailed evaluation is intentionally deferred to Week 4, while hyperparameter optimization is deferred to Week 5.
